In [386]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.impute import SimpleImputer
from sklearn.svm import SVC
from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV, StratifiedKFold
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score

In [387]:
df = pd.read_csv('../patient2025 - patient2025.csv')
df.head()

,HN,FBS,BMI,Diabetes,Chorestorol,age,hypertension,"vegetarian (1= yes, 0=no)",Marriage Status,Exercise (min/week),Living Area,stroke
0,11223,228.69,34.0,yes,201-220,71.0,1,1,Yes,0.0,Bangkok,1
1,8887,202.21,NaN,No,180-200,52.0,1,0,Yes,90.0,Country,1
2,5666,105.92,30.5,yes,180-200,78.0,1,1,Yes,0.0,Country,1
3,460182,171.23,35.0,No,221-260,54.0,1,0,Yes,0.0,Bangkok,1
4,166665,174.12,28.0,No,180-200,79.0,1,0,Yes,90.0,Country,1


In [388]:
df = df.drop(columns = ['HN', 'Marriage Status', 'Living Area'])
df

,FBS,BMI,Diabetes,Chorestorol,age,hypertension,"vegetarian (1= yes, 0=no)",Exercise (min/week),stroke
0,228.69,34.0,yes,201-220,71.00,1,1,0.0,1
1,202.21,NaN,No,180-200,52.00,1,0,90.0,1
2,105.92,30.5,yes,180-200,78.00,1,1,0.0,1
3,171.23,35.0,No,221-260,54.00,1,0,0.0,1
4,174.12,28.0,No,180-200,79.00,1,0,90.0,1
...,...,...,...,...,...,...,...,...,...
995,90.51,18.9,yes,Unknown,1.40,0,0,120.0,0
996,118.87,16.3,yes,Unknown,0.24,0,0,120.0,0
997,56.42,31.8,yes,180-200,55.00,0,0,0.0,0
998,73.67,21.0,No,Unknown,29.00,0,0,0.0,0


In [389]:
df["Diabetes"] = [1 if x == "yes" else 0 for x in df["Diabetes"]]
df

,FBS,BMI,Diabetes,Chorestorol,age,hypertension,"vegetarian (1= yes, 0=no)",Exercise (min/week),stroke
0,228.69,34.0,1,201-220,71.00,1,1,0.0,1
1,202.21,NaN,0,180-200,52.00,1,0,90.0,1
2,105.92,30.5,1,180-200,78.00,1,1,0.0,1
3,171.23,35.0,0,221-260,54.00,1,0,0.0,1
4,174.12,28.0,0,180-200,79.00,1,0,90.0,1
...,...,...,...,...,...,...,...,...,...
995,90.51,18.9,1,Unknown,1.40,0,0,120.0,0
996,118.87,16.3,1,Unknown,0.24,0,0,120.0,0
997,56.42,31.8,1,180-200,55.00,0,0,0.0,0
998,73.67,21.0,0,Unknown,29.00,0,0,0.0,0


In [390]:
df['Chorestorol'] = [np.nan if x == "Unknown" else x for x in df['Chorestorol']]

df[["chol_low", "chol_high"]] = (
    df["Chorestorol"]
    .str.split("-", expand=True)
)

df[["chol_low", "chol_high"]] = df[["chol_low", "chol_high"]].astype(float)

df["Chorestorol"] = df[["chol_low", "chol_high"]].mean(axis=1)
df.drop(columns=["chol_low", "chol_high"], inplace=True)
df

,FBS,BMI,Diabetes,Chorestorol,age,hypertension,"vegetarian (1= yes, 0=no)",Exercise (min/week),stroke
0,228.69,34.0,1,210.5,71.00,1,1,0.0,1
1,202.21,NaN,0,190.0,52.00,1,0,90.0,1
2,105.92,30.5,1,190.0,78.00,1,1,0.0,1
3,171.23,35.0,0,240.5,54.00,1,0,0.0,1
4,174.12,28.0,0,190.0,79.00,1,0,90.0,1
...,...,...,...,...,...,...,...,...,...
995,90.51,18.9,1,NaN,1.40,0,0,120.0,0
996,118.87,16.3,1,NaN,0.24,0,0,120.0,0
997,56.42,31.8,1,190.0,55.00,0,0,0.0,0
998,73.67,21.0,0,NaN,29.00,0,0,0.0,0


## ทดลอง 1
ทดลองใช้ Kernel ที่เป็น Linear โดย มีการหาค่า C และ Gamma ที่ดีที่สุด

In [391]:
features = ['FBS', 'BMI', 'age', 'Chorestorol', 'Diabetes', 'hypertension', 'vegetarian (1= yes, 0=no)', 'Exercise (min/week)']

for i in range(len(features) - 1):
    for j in range(i + 1, len(features)):
        subfeatures = [features[i], features[j]]
        X = df[subfeatures]
        y = df.stroke

        X_train, X_test, y_train, y_test = train_test_split(
            X, y,
            test_size=0.3,
            random_state=6
        )

        pipeline = Pipeline([
            ("imputer", SimpleImputer(strategy="mean")),
            ("scaler", StandardScaler()),
            ("svm", SVC(kernel="linear"))
        ])

        param_grid = {
            "svm__C" : [0.01, 0.1, 1, 10],
        }

        cv = StratifiedKFold(
            n_splits=5,
            shuffle=True,
            random_state=10
        )

        grid=GridSearchCV(
            pipeline,
            param_grid=param_grid,
            cv=cv,
            scoring="accuracy",
            n_jobs=-1
        )

        grid.fit(X_train, y_train)
        best_model = (grid.best_estimator_)
        predict_test = best_model.predict(X_test)
        print(best_model.score(X_train, y_train))
        print(accuracy_score(y_test, predict_test))
        print(classification_report(y_test, predict_test))
        print('----')

0.7557142857142857
0.74
              precision    recall  f1-score   support

           0       0.74      1.00      0.85       222
           1       0.00      0.00      0.00        78

    accuracy                           0.74       300
   macro avg       0.37      0.50      0.43       300
weighted avg       0.55      0.74      0.63       300

----
0.7942857142857143
0.82
              precision    recall  f1-score   support

           0       0.86      0.91      0.88       222
           1       0.69      0.56      0.62        78

    accuracy                           0.82       300
   macro avg       0.77      0.74      0.75       300
weighted avg       0.81      0.82      0.81       300

----
0.7557142857142857
0.74
              precision    recall  f1-score   support

           0       0.74      1.00      0.85       222
           1       0.00      0.00      0.00        78

    accuracy                           0.74       300
   macro avg       0.37      0.50      0.43   

c:\anaconda3\Lib\site-packages\sklearn\metrics\_classification.py:1531: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
c:\anaconda3\Lib\site-packages\sklearn\metrics\_classification.py:1531: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
c:\anaconda3\Lib\site-packages\sklearn\metrics\_classification.py:1531: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
c:\anaconda3\Lib\site-packages\sklearn\metrics\_classification.py:1531: Unde

0.7557142857142857
0.74
              precision    recall  f1-score   support

           0       0.74      1.00      0.85       222
           1       0.00      0.00      0.00        78

    accuracy                           0.74       300
   macro avg       0.37      0.50      0.43       300
weighted avg       0.55      0.74      0.63       300

----
0.7557142857142857
0.74
              precision    recall  f1-score   support

           0       0.74      1.00      0.85       222
           1       0.00      0.00      0.00        78

    accuracy                           0.74       300
   macro avg       0.37      0.50      0.43       300
weighted avg       0.55      0.74      0.63       300

----
0.7671428571428571
0.7566666666666667
              precision    recall  f1-score   support

           0       0.77      0.95      0.85       222
           1       0.59      0.22      0.32        78

    accuracy                           0.76       300
   macro avg       0.68      0.5

c:\anaconda3\Lib\site-packages\sklearn\metrics\_classification.py:1531: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
c:\anaconda3\Lib\site-packages\sklearn\metrics\_classification.py:1531: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
c:\anaconda3\Lib\site-packages\sklearn\metrics\_classification.py:1531: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
c:\anaconda3\Lib\site-packages\sklearn\metrics\_classification.py:1531: Unde

0.7871428571428571
0.81
              precision    recall  f1-score   support

           0       0.85      0.90      0.87       222
           1       0.66      0.56      0.61        78

    accuracy                           0.81       300
   macro avg       0.76      0.73      0.74       300
weighted avg       0.80      0.81      0.81       300

----
0.7557142857142857
0.74
              precision    recall  f1-score   support

           0       0.74      1.00      0.85       222
           1       0.00      0.00      0.00        78

    accuracy                           0.74       300
   macro avg       0.37      0.50      0.43       300
weighted avg       0.55      0.74      0.63       300

----
0.7557142857142857
0.74
              precision    recall  f1-score   support

           0       0.74      1.00      0.85       222
           1       0.00      0.00      0.00        78

    accuracy                           0.74       300
   macro avg       0.37      0.50      0.43   

...

c:\anaconda3\Lib\site-packages\sklearn\metrics\_classification.py:1531: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
c:\anaconda3\Lib\site-packages\sklearn\metrics\_classification.py:1531: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
c:\anaconda3\Lib\site-packages\sklearn\metrics\_classification.py:1531: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
c:\anaconda3\Lib\site-packages\sklearn\metrics\_classification.py:1531: Unde

0.7557142857142857
0.74
              precision    recall  f1-score   support

           0       0.74      1.00      0.85       222
           1       0.00      0.00      0.00        78

    accuracy                           0.74       300
   macro avg       0.37      0.50      0.43       300
weighted avg       0.55      0.74      0.63       300

----
0.7671428571428571
0.7566666666666667
              precision    recall  f1-score   support

           0       0.77      0.95      0.85       222
           1       0.59      0.22      0.32        78

    accuracy                           0.76       300
   macro avg       0.68      0.58      0.58       300
weighted avg       0.73      0.76      0.71       300

----
0.7557142857142857
0.74
              precision    recall  f1-score   support

           0       0.74      1.00      0.85       222
           1       0.00      0.00      0.00        78

    accuracy                           0.74       300
   macro avg       0.37      0.5

c:\anaconda3\Lib\site-packages\sklearn\metrics\_classification.py:1531: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
c:\anaconda3\Lib\site-packages\sklearn\metrics\_classification.py:1531: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
c:\anaconda3\Lib\site-packages\sklearn\metrics\_classification.py:1531: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
c:\anaconda3\Lib\site-packages\sklearn\metrics\_classification.py:1531: Unde

0.7557142857142857
0.74
              precision    recall  f1-score   support

           0       0.74      1.00      0.85       222
           1       0.00      0.00      0.00        78

    accuracy                           0.74       300
   macro avg       0.37      0.50      0.43       300
weighted avg       0.55      0.74      0.63       300

----
0.7671428571428571
0.7566666666666667
              precision    recall  f1-score   support

           0       0.77      0.95      0.85       222
           1       0.59      0.22      0.32        78

    accuracy                           0.76       300
   macro avg       0.68      0.58      0.58       300
weighted avg       0.73      0.76      0.71       300

----


c:\anaconda3\Lib\site-packages\sklearn\metrics\_classification.py:1531: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
c:\anaconda3\Lib\site-packages\sklearn\metrics\_classification.py:1531: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
c:\anaconda3\Lib\site-packages\sklearn\metrics\_classification.py:1531: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


## ทดลอง 2
ใช้ Kernel แบบ RBF โดยมีการทดลองหาค่า C และ Gamma ที่ดีที่สุด

In [392]:
features = ['FBS', 'BMI', 'age', 'Chorestorol', 'Diabetes', 'hypertension', 'vegetarian (1= yes, 0=no)', 'Exercise (min/week)']

X = df[features]
y = df.stroke

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.3,
    random_state=6
)

pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="mean")),
    ("scaler", StandardScaler()),
    ("svm", SVC(kernel="rbf"))
])

param_grid = {
    "svm__C": [0.01, 0.1, 1, 10, 100],
    "svm__gamma": [0.001, 0.01, 0.1, 1]
}

cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=10
)

grid = GridSearchCV(
    pipeline,
    param_grid,
    cv=cv,
    scoring="accuracy",
    n_jobs=-1
)

grid.fit(X_train, y_train)

best_model = grid.best_estimator_

predict_test = best_model.predict(X_test)

print(best_model.score(X_train, y_train))

print("Test score:", accuracy_score(y_test, predict_test))
print(classification_report(y_test, predict_test))

0.7957142857142857
Test score: 0.78
              precision    recall  f1-score   support

           0       0.81      0.92      0.86       222
           1       0.62      0.38      0.48        78

    accuracy                           0.78       300
   macro avg       0.72      0.65      0.67       300
weighted avg       0.76      0.78      0.76       300



## ทดลอง 3
ใช้ Kernel แบบ Polynomial โดยมีการทดลองหาค่า C, Gamma และ coef0 ที่ดีที่สุด และกำหนด Degree ให้เป็น 2

In [393]:
features = ['FBS', 'BMI', 'age', 'Chorestorol', 'Diabetes', 'hypertension', 'vegetarian (1= yes, 0=no)', 'Exercise (min/week)']

X = df[features]
y = df.stroke

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.3,
    random_state=6
)

pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="mean")),
    ("scaler", StandardScaler()),
    ("svm", SVC(degree=2, kernel="poly"))
])

param_grid = {
    "svm__C": [0.01, 0.1, 1, 10, 100],
    "svm__gamma": [0.001, 0.01, 0.1, 1],
    "svm__coef0": [0, 0.1, 1, 10]
}

cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=10
)

grid = GridSearchCV(
    pipeline,
    param_grid,
    cv=cv,
    scoring="accuracy",
    n_jobs=-1
)

grid.fit(X_train, y_train)

best_model = grid.best_estimator_

predict_test = best_model.predict(X_test)

print(best_model.score(X_train, y_train))

print("Test score:", accuracy_score(y_test, predict_test))
print(classification_report(y_test, predict_test))

0.8085714285714286
Test score: 0.7866666666666666
              precision    recall  f1-score   support

           0       0.79      0.96      0.87       222
           1       0.73      0.28      0.41        78

    accuracy                           0.79       300
   macro avg       0.76      0.62      0.64       300
weighted avg       0.78      0.79      0.75       300



## ทดลอง 4
ใช้ Kernel แบบ Sigmoid โดยมีการทดลองหาค่า C, Gamma และ coef0 ที่ดีที่สุด

In [394]:
features = ['FBS', 'BMI', 'age', 'Chorestorol', 'Diabetes', 'hypertension', 'vegetarian (1= yes, 0=no)', 'Exercise (min/week)']

X = df[features]
y = df.stroke

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.3,
    random_state=6
)

pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="mean")),
    ("scaler", StandardScaler()),
    ("svm", SVC(kernel="sigmoid"))
])

param_grid = {
    "svm__C": [0.01, 0.1, 1, 10, 100],
    "svm__gamma": [0.001, 0.01, 0.1, 1],
    "svm__coef0": [0, 0.1, 1, 10]
}

cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=10
)

grid = GridSearchCV(
    pipeline,
    param_grid,
    cv=cv,
    scoring="accuracy",
    n_jobs=-1
)

grid.fit(X_train, y_train)

best_model = grid.best_estimator_

predict_test = best_model.predict(X_test)

print(best_model.score(X_train, y_train))

print("Test score:", accuracy_score(y_test, predict_test))
print(classification_report(y_test, predict_test))

0.7742857142857142
Test score: 0.7833333333333333
              precision    recall  f1-score   support

           0       0.81      0.92      0.86       222
           1       0.63      0.40      0.49        78

    accuracy                           0.78       300
   macro avg       0.72      0.66      0.68       300
weighted avg       0.77      0.78      0.77       300

